# Fine-Tune dataset using yolo11s

**Перед нічним запуском:**

1. У терміналі запустити `caffeinate -is` (і не закривати його) — інакше macOS засне і тренування зупиниться.
2. Тримати Mac на зарядці, вікно VS Code не закривати.
3. Якщо запуск перервався — продовжити з останнього чекпоінта:
   ```python
   model = YOLO("../runs/baseline_yolo11s_e30/weights/last.pt")
   model.train(resume=True)
   ```

Результати: `../runs/baseline_yolo11s_e30/` — `results.csv` (метрики по епохах), криві PR/F1, confusion matrix, `weights/best.pt`.

> Нагадування з EDA: через витік train/val (26/27 спільних сесій) метрики на val завищені — фінальну оцінку робити на test.

## Чесна оцінка: витік train/val і test-baseline

`best.pt` після 30 епох показував на val mAP50=0.98, але на predict з нового відео — жахливо. Причина: 26 з 27 train-сесій присутні і в val (спільний фон/дрон/траєкторія, різні лише кліпи), тобто val-метрики були завищені через витік даних. `test/` — єдиний спліт без перетину на рівні сесій, і саме на ньому видно реальну картину.

**Зроблено:**
1. `framecut.py` + `src/dataset.py` нарізали і сконвертували `test/` у YOLO-формат (15946 кадрів, 14787 боксів) → `yolo_dataset/images|labels/test`.
2. `src/resplit_by_session.py` перебудував train/val: сесії розбиваються цілком в один спліт (seed=0, val_frac=0.2) → 21 train-сесій / 6 val-сесій, **0 спільних сесій** (було 26/27).


In [ ]:
metrics_test = model.val(
    data="../Anti-UAV-RGBT/yolo_dataset/data.yaml",
    split="test",
    device="mps",
)
print(metrics_test)


**Реальний baseline (best.pt, 30 епох, старий leaky train) на чистому test:**

| метрика | val (leaky) | test (чистий) |
|---|---|---|
| precision | 0.991 | 0.962 |
| recall | 0.977 | 0.825 |
| mAP50 | 0.982 | 0.866 |
| mAP50-95 | 0.587 | 0.449 |

Модель не «жахлива» — mAP50 0.87 і precision 0.96 на невіданих сценах це непогано. Основна слабкість — **recall 0.825**: ~17.5% дронів на нових сценах пропускаються. Розрив із val (0.98→0.87 mAP50, 0.977→0.825 recall) підтверджує, що val був завищений через витік, а не що модель раптом стала гіршою.

**Далі:** тренувати на новому train/val (без витоку сесій) — тепер mAP на val нарешті відображатиме реальну узагальнювальну здатність, і `patience=10` матиме сенс. Дотренування давнього `best.pt` ще на 20 епох на leaky-спліті сенсу не має — його val-крива все одно нічого чесного не покаже.

In [2]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

train_results = model.train(
    data="../Anti-UAV-RGBT/yolo_dataset/data.yaml",
    epochs=30,
    patience=10, 
    imgsz=640,
    device="mps",
    workers=4,
    cache=False,
    optimizer="Adam",
    project="../runs",
    name="baseline_yolo11s_e30",
    save_period=5,  
)

metrics = model.val()
print(metrics)

Ultralytics 8.4.104 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../Anti-UAV-RGBT/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yolo11s_e30, nbs=64, nms=False, opset=None, opt

/Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic s

       1/30      8.63G       1.71     0.9595      1.244          9        640: 100% ━━━━━━━━━━━━ 1051/1051 1.2s/it 21:151.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 546/546 1.5it/s 5:540.7ss
                   all      17468      16065       0.89      0.765      0.812      0.333

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic s

       2/30      7.63G      1.647     0.8547      1.214         14        640: 100% ━━━━━━━━━━━━ 1051/1051 1.2s/it 20:411.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 546/546 1.5it/s 5:570.7ss
                   all      17468      16065      0.953      0.938      0.953      0.483

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      28/30      7.63G      1.348     0.4918      1.145          9        640: 100% ━━━━━━━━━━━━ 1051/1051 1.2s/it 20:151.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 546/546 1.5it/s 5:520.7ss
                   all      17468      16065      0.991      0.977      0.982      0.581

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic s

      29/30      7.63G      1.342     0.4882      1.139          8        640: 100% ━━━━━━━━━━━━ 1051/1051 1.2s/it 20:281.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 546/546 1.5it/s 5:580.7ss
                   all      17468      16065       0.99      0.977      0.982      0.586

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/dmytroborbotko/ai-projects/real-time-uav-detection/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic s

      30/30      7.63G      1.335     0.4816      1.138          8        640: 100% ━━━━━━━━━━━━ 1051/1051 1.2s/it 20:241.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 546/546 1.5it/s 5:570.7ss
                   all      17468      16065      0.991      0.977      0.982      0.587

30 epochs completed in 13.266 hours.
Optimizer stripped from /Users/dmytroborbotko/ai-projects/real-time-uav-detection/runs/runs/baseline_yolo11s_e30/weights/last.pt, 19.2MB
Optimizer stripped from /Users/dmytroborbotko/ai-projects/real-time-uav-detection/runs/runs/baseline_yolo11s_e30/weights/best.pt, 19.2MB

Validating /Users/dmytroborbotko/ai-projects/real-time-uav-detection/runs/runs/baseline_yolo11s_e30/weights/best.pt...
Ultralytics 8.4.104 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M1 Pro)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
                 Class     Images  Instances      Box(P          R 